## ListView basado en clases protegido por Login

### Editamos el modelo para añadir una llave foranea 

#### products/models.py

In [ ]:
from django.conf import settings #<--------nuevo
from django.db import models



User = settings.AUTH_USER_MODEL  #<---------nuevo

class Product(models.Model):
    user = models.ForeignKey(User , blank=True, null=True, on_delete=models.SET_NULL) #<--------nuevo
    title = models.CharField(max_length=120)
    slug = models.SlugField(unique=True)

#### Hacemos la migracion 

- python manage.py makemigrations
- python manage.py migrate 

### Creamos la vista

#### products/views.py

In [ ]:
from django.contrib.auth.mixins import LoginRequiredMixin
from django.views.generic  import (
    ListView,
    DetailView,
    RedirectView,
    )
from django.shortcuts import render , get_object_or_404

from .forms import ProductModelForm

from .mixins import TemplateTitleMixin 
from .models import Product , DigitalProduct


# Creamos la vista protegida por loginrequiredmixin

class ProtectedListView(LoginRequiredMixin, ListView):
    model = Product
    template_name = "products/product_list.html"

    
# Sobrescribimos el metodo para que filtre por usuario  
    
    def get_queryset(self):
        return Product.objects.filter(user=self.request.user)


### Hacemos el template
Nota: yo ya lo tenia hecho pero se hace si no esta

#### products/products/product_list.html

In [ ]:
{% if title %}
    <h1> {{ title }} </h1>
{% endif %}


{% for object in object_list %}

    <li>{{ object.title}} - {{ object.slug}}</li>

{% endfor %}

### Añadir la url 

#### products/urls.py

In [ ]:
from django.contrib import admin
from django.urls import path  
from django.views.generic import TemplateView ,RedirectView

from products import views
from products.views import (
    ProductListView,
    ProductDetailView , 
    DigitalProduct , 
    ProtectedListView,#<------------ Se importo 
    ProtectedProductDetailView,
    ProductIDRedirectView,
    ProductRedirectView,
)

urlpatterns = [ 
    path("admin/" , admin.site.urls),
    path("aboout-us", RedirectView.as_view(url="/products/about/")),
    path("about/", TemplateView.as_view(template_name="about.html")),
    path("team/", TemplateView.as_view(template_name="team.html")),
    path("products/", ProductListView.as_view()),
    path("digital-products/", DigitalProduct.as_view()),
    path("products/<int:pk>", ProductDetailView.as_view()),
    path("products/<slug:slug>",ProductDetailView.as_view()),
    path("p/<int:pk>", ProductIDRedirectView.as_view()),
    path("p/<slug:slug>",ProductRedirectView.as_view()),
    path("my-products/",ProtectedListView.as_view()),# <----------------------------- Se añadio
    path("my-products/<slug:slug>",ProtectedProductDetailView.as_view()),
]
